In [ ]:
import gymnasium as gym
import numpy as np
import subprocess
import os
import wave
from gymnasium.wrappers import RecordVideo
from gymnasium.wrappers import ResizeObservation
from sdlarch_rl import make
import pygame
import cv2
import time

import numpy as np

os.makedirs("videos", exist_ok=True)


video_filename = "videos/video-episode-0.mp4"
audio_filename = "videos/game_audio.wav"

audio_data = b""
arate = None
framerate = None

data = {
    "audio_data": audio_data,
    "arate": arate,
    "framerate": framerate,
}

frames = []
# env = make("GranTurismo3-Ps2", env_variables=[{"pcsx2_upscale_multiplier", "3x Native (~1080p)"}])
#env = make("NewSuperMarioBros-Wii", env_variables=[{"dolphin_efb_scale", "x2 (1280 x 1056)"}])
# env = make("MarioDash-GC", env_variables=[{"dolphin_efb_scale", "x2 (1280 x 1056)"}])
# env = make("VirtuaTennis-DC")
#env = make("MarvelVsCapcom2-DC")
# env = make("SuperMario64-N64")
env = make("YoshiIsland-NDS")
# env = make("GTASanAndreas-Ps2", env_variables=[{"pcsx2_upscale_multiplier", "3x Native (~1080p)"}])

# env = RecordVideo(
#     env,
#     video_folder="videos/",
#     episode_trigger=lambda x: x == 0,
#     name_prefix="video"
# )


render_mode = "rgb_array"
# render_mode = "human"


obs, info = env.reset()
done = False
count = 0

pygame.init()

SCREEN_WIDTH = 1920
SCREEN_HEIGHT = 1080
window = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT))
clock = pygame.time.Clock()

data['arate'] = env.unwrapped.em.get_audio_rate()
data['framerate'] = env.unwrapped.em.get_frame_rate()

print("arate: ", data['arate'])
print("framerate: ", data['framerate'])

frame_time = 1.0 / data['framerate']
last_time = time.time()

while True:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            done = True
            
    keys = pygame.key.get_pressed()

    env.render()

    action = np.zeros(16, dtype=np.uint8)

    if keys[pygame.K_UP]:
        action[4] = 1
    if keys[pygame.K_DOWN]:
        action[5] = 1
    if keys[pygame.K_LEFT]:
        action[6] = 1
    if keys[pygame.K_RIGHT]:
        action[7] = 1
    if keys[pygame.K_c]:
        action[1] = 1
    if keys[pygame.K_x]:
        action[0] = 1
    if keys[pygame.K_RETURN]:
        action[3] = 1
    if keys[pygame.K_l]:
        action[11] = 1

    # clock.tick(60)

    img, rew, done, _, info = env.step(action)

    sound = env.unwrapped.em.get_audio()
    #print(sound.shape)
    data['audio_data'] += sound.tobytes()

     # stop
    if keys[pygame.K_BACKSPACE]:
        print("== DONE ==")
        done = True

    # frame rate
    now = time.time()
    sleep_time = frame_time - (now - last_time)
    if sleep_time > 0:
        time.sleep(sleep_time)
    last_time = now

    img = cv2.resize(img, (SCREEN_WIDTH, SCREEN_HEIGHT))

    if img is not None:
        frame = cv2.resize(img, (640, 480))
        frames.append(frame)
     
    surface = pygame.surfarray.make_surface(np.transpose(img, (1, 0, 2)))


    window.blit(surface, (0, 0))
    pygame.display.update()

    count += 1

    if done:
        break

env.close()
pygame.quit()

out = cv2.VideoWriter(video_filename, cv2.VideoWriter_fourcc(*"mp4v"), 60, (640, 480))
for f in frames:
    out.write(cv2.cvtColor(f, cv2.COLOR_RGB2BGR))
out.release()

print("arate: ", data['arate'])
print("framerate: ", data['framerate'])

# save audio
with wave.open(audio_filename, "wb") as wf:
    wf.setnchannels(2)  # Mono
    wf.setsampwidth(2)  # 16-bit PCM
    wf.setframerate(data['arate'])  # Audio rate
    wf.writeframes(data['audio_data'])

video_prefix = "video.mp4"


ffmpeg_cmd = [
    "ffmpeg", "-y",
    "-r", str(data['framerate']),
    "-i", video_filename,
    "-i", audio_filename,
    # "-vf", "scale=1280:720", # hd resolution
    "-vf", f"scale={SCREEN_WIDTH}:{SCREEN_HEIGHT},setdar=16:9,setsar=1", # HD
    # "-vf", "scale=854:480,setdar=16:9,setsar=1", # 16x9
    "-c:v", "libx264",
    # "-preset", "veryslow",
     "-crf", "10", # quality 10 ~ 50 (10 is better)
    "-c:a", "aac",
    "-b:a", "128k",
    "-ac","2",
    "-map", "0:v:0",       
    "-map", "1:a:0",
    "-strict", "experimental",
    "-shortest",
    "./videos/" + video_prefix
]

subprocess.run(ffmpeg_cmd)

print("✅ Finished vídeos")

Detected game: yoshiisland-nds
Variable: desmume_firmware_language = Auto

Variable: desmume_use_external_bios = disabled

Variable: desmume_boot_into_bios = disabled

Variable: desmume_load_to_memory = disabled

Variable: desmume_num_cores = 1

Variable: desmume_cpu_mode = interpreter

Variable: desmume_jit_block_size = 12

Variable: desmume_advanced_timing = enabled

Variable: desmume_frameskip = 0

Variable: desmume_internal_resolution = 256x192

Variable: desmume_opengl_mode = enabled

Variable: desmume_color_depth = 16-bit

Variable: desmume_gfx_multisampling = disabled

Variable: desmume_gfx_texture_smoothing = disabled

Variable: desmume_opengl_shadow_polygon = enabled

Variable: desmume_opengl_special_zero_alpha = enabled

Variable: desmume_opengl_nds_depth_calculation = enabled

Variable: desmume_opengl_depth_lequal_polygon_facing = disabled

Variable: desmume_gfx_highres_interpolate_color = disabled

Variable: desmume_gfx_linehack = enabled

Variable: desmume_gfx_txthack = di